In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# CONFIG
# ==========================

FILE1 = "/content/drive/MyDrive/fp/final_predictions Krishna's input.csv"
FILE2 = "/content/drive/MyDrive/fp/final_predictions_Bhumika's input.csv"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ================================================================
# LEGAL FACT LAYER — GROQ EDITION
# Output: one JSON file per case in output folder
# ================================================================

# ── CELL 1: Install ──────────────────────────────────────────

!pip install -q groq pandas tqdm

In [ ]:
# ── CELL 2: Imports ──────────────────────────────────────────

import pandas as pd

import re
import json
import os
import time
import gc
import traceback
import hashlib                     # fix: collision-proof case IDs
from datetime import datetime      # fix: date normalization helpers

from tqdm.auto import tqdm
from groq import Groq

print("✅ Imports OK")

✅ Imports OK


In [ ]:
# ── CELL 3: Config — ONLY EDIT THIS CELL ─────────────────────

GROQ_API_KEY = "add-key-here"
FILE_PATH      = "/content/drive/MyDrive/fp/final_predictions Krishna's input.csv"
OUTPUT_DIR     = "/content/drive/MyDrive/fact_layer_cases_v2"
CHECKPOINT_DIR = "/content/drive/MyDrive/fact_layer_cases_v2/checkpoints"


PAUSE_SECS  = 2.2
MAX_RETRIES = 4
BATCH_SIZE  = 100
DAILY_LIMIT = 14000
RESUME      = True

# fix: Groq token-budget safety stop
MAX_TOKEN_BUDGET = 480000

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print(
    f"✅ Config OK | Cases will be saved to → {OUTPUT_DIR}"
)

✅ Config OK | Cases will be saved to → /content/drive/MyDrive/fact_layer_cases_v2


In [ ]:
# ── CELL 4: Groq client ──────────────────────────────────────

client = Groq(api_key=GROQ_API_KEY)

SYSTEM = """
You are a legal information extraction expert for Indian Supreme Court judgments.

Return ONLY valid JSON.

No markdown.
No code fences.
No explanation.
No commentary.

Important legal context:

- This is Indian legal jurisprudence.
- "Dismissed" has a precise procedural meaning.
- "Natural Justice" refers to doctrines such as audi alteram partem
  and nemo judex in causa sua.
- Constitutional provisions, legal doctrines and procedural outcomes
  must be interpreted using Indian legal usage.

Extraction Rules:

1. If information is not explicitly present in the text:
   return null or [].

2. Do NOT infer.

3. Do NOT fabricate.

4. Do NOT generate hypothetical facts.

5. Do NOT generate hypothetical legal arguments.

6. If fewer than 3 facts are present:
   return only the facts that exist.

7. If respondent arguments are missing:
   return [].

8. Facts means:
   real-world events that happened before court proceedings.

9. Do NOT include procedural history as facts.

10. Use complete sentences in all list fields.

11. Outcome must be EXACTLY one of:

    Accept
    Reject
    Partially Accept

12. Return valid JSON only.
"""

# ============================================================
# TOKEN ACCOUNTING
# ============================================================

total_tokens_used = 0


# ============================================================
# DAILY LIMIT EXCEPTION
# ============================================================

class DailyLimitReached(Exception):
    pass


# ============================================================
# OUTCOME NORMALIZATION
# ============================================================

OUTCOME_MAP = {

    # ACCEPT
    "accept": "Accept",
    "accepted": "Accept",
    "allow": "Accept",
    "allowed": "Accept",
    "granted": "Accept",
    "quashed": "Accept",
    "set aside": "Accept",

    # REJECT
    "reject": "Reject",
    "rejected": "Reject",
    "dismiss": "Reject",
    "dismissed": "Reject",
    "upheld": "Reject",

    # PARTIAL
    "partial": "Partially Accept",
    "partially accept": "Partially Accept",
    "partially accepted": "Partially Accept",
    "partially allowed": "Partially Accept",
    "partly allowed": "Partially Accept",
    "modified": "Partially Accept",

    # SPECIAL
    "withdrawn": "Withdrawn",
}


def normalize_outcome(outcome):

    if not outcome:
        return None

    outcome = str(outcome).strip().lower()

    if outcome in OUTCOME_MAP:
        return OUTCOME_MAP[outcome]

    if any(
        x in outcome
        for x in [
            "partially allowed",
            "partly allowed",
            "modified",
            "partial"
        ]
    ):
        return "Partially Accept"

    if any(
        x in outcome
        for x in [
            "allowed",
            "accepted",
            "granted",
            "quashed",
            "set aside"
        ]
    ):
        return "Accept"

    if any(
        x in outcome
        for x in [
            "dismissed",
            "rejected",
            "upheld"
        ]
    ):
        return "Reject"

    return None


# ============================================================
# RATE LIMIT HANDLER
# ============================================================

def _handle_rate_limit(exc, retry_no):

    err = str(exc).lower()

    # --------------------------------------------------------
    # DAILY TOKEN LIMIT
    # --------------------------------------------------------

    if (
        "tokens per day" in err
        or "daily token" in err
        or "daily limit" in err
        or "per day" in err
    ):

        print(
            "\n🛑 Daily Groq token limit reached."
            "\nResume tomorrow."
        )

        raise DailyLimitReached(err)

    # --------------------------------------------------------
    # MINUTE TOKEN LIMIT
    # --------------------------------------------------------

    if (
        "tokens per minute" in err
        or "rate_limit_tokens" in err
        or "rate_limit" in err
        or "429" in err
    ):

        wait = 65

        print(
            f"\n⏳ Minute limit reached."
            f"\nWaiting {wait} seconds..."
        )

        time.sleep(wait)

        return

    # --------------------------------------------------------
    # GENERIC BACKOFF
    # --------------------------------------------------------

    wait = 5 * (retry_no + 1)

    print(
        f"\n⚠️ Retrying in {wait}s"
    )

    time.sleep(wait)


# ============================================================
# GROQ CALL
# ============================================================

def call_groq(prompt: str):

    """
    Returns:

        (parsed_json, "groq")

    OR

        (None, "regex_fallback")
    """

    global total_tokens_used

    if total_tokens_used >= MAX_TOKEN_BUDGET:

        raise DailyLimitReached(
            "Local token budget exhausted"
        )

    attempts = [

        # Normal extraction
        (
            0.05,
            ""
        ),

        # JSON repair pass
        (
            0.0,
            (
                "Your previous response was not valid JSON.\n"
                "Return ONLY a valid JSON object.\n"
                "Start with { and end with }.\n\n"
            )
        ),
    ]

    for attempt_index, (temperature, prefix) in enumerate(attempts):

        for retry in range(MAX_RETRIES):

            try:

                resp = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {
                            "role": "system",
                            "content": SYSTEM
                        },
                        {
                            "role": "user",
                            "content": prefix + prompt
                        }
                    ],
                    temperature=temperature,
                    max_tokens=1400
                )

                try:
                    total_tokens_used += (
                        resp.usage.total_tokens
                    )
                except:
                    pass

                raw = (
                    resp.choices[0]
                    .message
                    .content
                    .strip()
                )

                raw = re.sub(
                    r"^```(?:json)?\s*",
                    "",
                    raw
                )

                raw = re.sub(
                    r"\s*```$",
                    "",
                    raw
                )

                parsed = json.loads(raw)

                time.sleep(PAUSE_SECS)

                return parsed, "groq"

            except json.JSONDecodeError:

                if attempt_index == 0:
                    break

                return (
                    None,
                    "regex_fallback"
                )

            except DailyLimitReached:
                raise

            except Exception as e:

                _handle_rate_limit(
                    e,
                    retry
                )

    return (
        None,
        "regex_fallback"
    )


print("✅ Groq client ready")

✅ Groq client ready


In [ ]:
# ── CELL 5: Prompt ───────────────────────────────────────────

def build_prompt(
    facts,
    petitioner_args,
    respondent_args,
    reasoning,
    decision,
    title_hint,
    year,
    issue_hint=None
):

    issue_block = ""

    if issue_hint:
        issue_block = f"""
POSSIBLE ISSUE IDENTIFIED FROM JUDGMENT:

{issue_hint}

Verify this issue statement and refine it into one precise legal issue.
"""

    return f"""
You are extracting structured information from an Indian Supreme Court judgment.

IMPORTANT LEGAL CONTEXT:

- This is Indian legal jurisprudence.
- "Dismissed" has a procedural legal meaning.
- "Natural Justice" refers to doctrines such as:
  audi alteram partem and nemo judex in causa sua.
- Constitutional provisions, doctrines and tests should be
  interpreted using Indian legal terminology.

============================================================
CRITICAL OUTPUT RULE
============================================================

Outcome MUST be EXACTLY one of:

Accept
Reject
Partially Accept

No other values are allowed.

============================================================
CASE INFORMATION
============================================================

Case Year: {year}

Filename Hint:
{title_hint}

{issue_block}

============================================================
FACTS
============================================================

{facts[:1800]}

============================================================
PETITIONER ARGUMENTS
============================================================

{petitioner_args[:1200]}

============================================================
RESPONDENT ARGUMENTS
============================================================

{respondent_args[:1200]}

============================================================
REASONING
============================================================

{reasoning[:1800]}

============================================================
DECISION
============================================================

{decision[:1200]}

============================================================
RETURN ONLY VALID JSON
============================================================

{{
  "title": "Formal case title",

  "court": "Supreme Court of India",

  "bench_strength": 5,

  "case_type": "Constitutional",

  "issue":
    "One sentence describing the precise legal issue",

  "facts": [
    "Fact 1",
    "Fact 2",
    "Fact 3"
  ],

  "petitioner_arguments": [
    "Argument 1",
    "Argument 2"
  ],

  "respondent_arguments": [
    "Argument 1",
    "Argument 2"
  ],

  "reasoning_principles": [
    "Legal doctrine",
    "Constitutional principle"
  ],

  "reasoning_summary":
    "Short explanation of how the Court reasoned.",

  "outcome":
    "Accept",

  "decision_details":
    "Specific order passed by the Court.",

  "outcome_reasoning":
    "One sentence explaining what the Court ordered and to whom.",

  "case_summary":
    "Why the case is important."
}}

============================================================
RULES
============================================================

1. Return ONLY JSON.

2. No markdown.

3. No explanation outside JSON.

4. If information is absent:
   use null or [].

5. Do NOT infer information.

6. Do NOT fabricate information.

7. Do NOT generate hypothetical facts.

8. Do NOT generate hypothetical arguments.

------------------------------------------------------------
FACTS
------------------------------------------------------------

Facts means:

REAL-WORLD EVENTS BEFORE ANY COURT FILING.

Facts should describe:

- what happened to the parties
- what occurred in the real world
- events giving rise to the dispute

Do NOT include:

- procedural history
- filing history
- lower court history
- appeals history
- court chronology

Bad Example:

"The Sessions Court convicted the accused."

Good Example:

"The accused was found in possession of the weapon."

------------------------------------------------------------
ISSUE
------------------------------------------------------------

Issue should be:

A single precise legal question the Court answered.

Use any issue explicitly stated in the judgment if present.

------------------------------------------------------------
ARGUMENTS
------------------------------------------------------------

Petitioner arguments must come only from petitioner material.

Respondent arguments must come only from respondent material.

If respondent arguments do not appear:

return []

Do NOT invent arguments.

------------------------------------------------------------
REASONING PRINCIPLES
------------------------------------------------------------

reasoning_principles must contain:

- legal doctrines
- constitutional doctrines
- statutory interpretations
- legal tests
- named principles

Examples:

- Basic Structure Doctrine
- Proportionality Test
- Reasonable Classification Test under Article 14
- Natural Justice
- Doctrine of Severability

Do NOT return:

- procedural observations
- factual observations
- generic comments

------------------------------------------------------------
OUTCOME
------------------------------------------------------------

Outcome MUST be EXACTLY one of:

Accept
Reject
Partially Accept

No other value allowed.

Examples:

Allowed → Accept

Granted → Accept

Quashed → Accept

Dismissed → Reject

Partly Allowed → Partially Accept

------------------------------------------------------------
MISSING INFORMATION
------------------------------------------------------------

If fewer than 3 facts appear:

return only available facts.

If no respondent arguments appear:

return [].

If court is unclear:

return null.

If bench strength is unclear:

return null.

Never invent information to make fields look complete.

============================================================
END
============================================================
"""

CELL 6 — PART 1

In [ ]:
# ── CELL 6: Regex helpers ────────────────────────────────────

# ============================================================
# DATE HELPERS
# ============================================================

_MONTHS = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,
}

_DATE_PATTERNS = [

    (
        r"(\d{1,2})(?:st|nd|rd|th)\s+([A-Za-z]+),?\s+(\d{4})",
        "dmy_ordinal"
    ),

    (
        r"([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})",
        "mdy_text"
    ),

    (
        r"DATED?\s+this\s+(\d{1,2})(?:st|nd|rd|th)\s+day\s+of\s+([A-Za-z]+)\s+(\d{4})",
        "formal"
    ),

    (
        r"(\d{2})[./\-](\d{2})[./\-](\d{4})",
        "dmy_numeric"
    ),

    (
        r"DATE\s+OF\s+JUDGMENT[:\s]+(\d{2}/\d{2}/\d{4})",
        "labeled"
    ),
]


def _to_iso(groups, fmt):

    try:

        if fmt in ("dmy_ordinal", "formal"):

            d, m, y = groups

            mn = _MONTHS.get(
                m.lower()
            )

            if not mn:
                return None

            return f"{y}-{mn:02d}-{int(d):02d}"

        if fmt == "mdy_text":

            m, d, y = groups

            mn = _MONTHS.get(
                m.lower()
            )

            if not mn:
                return None

            return f"{y}-{mn:02d}-{int(d):02d}"

        if fmt == "dmy_numeric":

            d, m, y = groups

            return f"{y}-{m}-{d}"

        if fmt == "labeled":

            d, m, y = groups[0].split("/")

            return f"{y}-{m}-{d}"

    except:
        return None

    return None


# ============================================================
# STATUTES
# ============================================================

_STATUTE_PATTERNS = [

    r"(?:Article|Art\.)\s*\d+[A-Z]?(?:\([a-z0-9]\))*(?:\s+of\s+(?:the\s+)?Constitution)?",

    r"[Ss]ection\s+\d+[A-Z]?\s+of\s+(?:the\s+)?[A-Z][^\n,;\.]{3,60}?(?:Act|Code|Ordinance)",

    r"(?:IPC|CrPC|CPC)\s+(?:[Ss]ection\s+)?\d+[A-Z]?",

    # fix
    r"[Uu]/[Ss]\s*\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))?\s+(?:IPC|CrPC|CPC|IEA)",

    # fix
    r"[Ss]\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))?\s+(?:IPC|CrPC|CPC|IEA)",

    # fix
    r"[Ss]ec(?:tion)?\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))*",

    # fix
    r"[Aa]rticle\s+\d+[A-Z]?\s*/\s*\d+[A-Z]?",

    # fix
    r"[Oo]rder\s+(?:[IVXLC]+|\d+)\s+[Rr]ule\s+\d+[A-Z]?\s+(?:CPC|CrPC)",

    # fix
    r"[Ss]chedule\s+(?:[IVXLC]+|\d+)\s+[Ee]ntry\s+\d+[A-Z]?",
]


def roman_to_int(s):

    vals = {
        "I":1,
        "V":5,
        "X":10,
        "L":50,
        "C":100
    }

    total = 0
    prev = 0

    for ch in reversed(s.upper()):

        cur = vals.get(ch,0)

        if cur < prev:
            total -= cur
        else:
            total += cur

        prev = cur

    return total


def _norm_statute(raw):

    raw = raw.strip()

    m = re.search(
        r"[Aa]rticle\s+(\d+[A-Z]?)\s*/\s*(\d+[A-Z]?)",
        raw
    )

    if m:

        return [
            f"Constitution_Art_{m.group(1)}",
            f"Constitution_Art_{m.group(2)}",
        ]

    m = re.search(
        r"[Aa]rticle\s+(\d+[A-Z]?(?:\([a-z0-9]+\))*)",
        raw
    )

    if m:
        return f"Constitution_Art_{m.group(1)}"

    m = re.search(
        r"(IPC|CrPC|CPC|IEA)\s+(?:[Ss]ection\s+)?(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:
        return f"{m.group(1)}_S{m.group(2)}"

    m = re.search(
        r"[Uu]/[Ss]\s*\.?\s*(\d+[A-Z]?)",
        raw
    )

    if m:
        return f"IPC_S{m.group(1)}"

    m = re.search(
        r"[Ss]\.?\s*(\d+[A-Z]?)",
        raw
    )

    if m:
        return f"IPC_S{m.group(1)}"

    m = re.search(
        r"[Ss]ec(?:tion)?\.?\s*(\d+[A-Z]?(?:\([a-z0-9]+\))*)",
        raw
    )

    if m:
        return f"Section_{m.group(1)}"

    m = re.search(
        r"[Oo]rder\s+([IVXLC]+|\d+)\s+[Rr]ule\s+(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:

        order = m.group(1)

        if order.isalpha():
            order = roman_to_int(order)

        return f"CPC_Order{order}_Rule{m.group(2)}"

    m = re.search(
        r"[Ss]chedule\s+([IVXLC]+|\d+)\s+[Ee]ntry\s+(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:

        sch = m.group(1)

        if sch.isalpha():
            sch = roman_to_int(sch)

        return (
            f"Constitution_Schedule{sch}"
            f"_Entry{m.group(2)}"
        )

    return None


def extract_statutes(text):

    seen = set()
    out = []

    for pat in _STATUTE_PATTERNS:

        for m in re.finditer(
            pat,
            text,
            re.I
        ):

            sid = _norm_statute(
                m.group(0)
            )

            if not sid:
                continue

            if isinstance(
                sid,
                list
            ):

                for x in sid:

                    if x not in seen:
                        seen.add(x)
                        out.append(x)

            else:

                if sid not in seen:
                    seen.add(sid)
                    out.append(sid)

    return out


# ============================================================
# CITATIONS
# ============================================================

_CITATION_PATTERNS = [

    r"\(\d{4}\)\s+\d+\s+SCC\s+\d+",

    r"\d{4}\s*\(\s*\d+\s*\)\s*SCC\s+\d+",

    r"\d{4}\s+Supp(?:l(?:ement)?)?\s+SCC\s+\d+",

    r"JT\s+\d{4}\s*\(\s*\d+\s*\)\s*SC\s+\d+",

    r"MANU/(?:SC|HC)/\d{1,6}/\d{4}",

    r"W\.?P\.?\s*\([A-Z]+\)\s*[Nn]o\.?\s*\d+\s*/\s*\d{4}",

    r"(?:Civil|Criminal)\s+Appeal\s+[Nn]o\.?\s*\d+\s+of\s+\d{4}",

    r"AIR\s+\d{4}\s+SC(?:\s+\d+)?",

    r"\d{4}\s+SCR\s+\d+",
]


def extract_citations(text):

    found = []

    for pat in _CITATION_PATTERNS:

        for m in re.finditer(
            pat,
            text,
            re.I
        ):

            c = re.sub(
                r"\s+",
                " ",
                m.group(0).strip()
            )

            if c not in found:
                found.append(c)

    return found

CELL 6 — PART 2 (Judges, Metadata, Parties, Concepts, Outcome)

In [ ]:
# ============================================================
# JUDGES EXTRACTION
# ============================================================

def _normalize_judge_name(name):

    name = re.sub(
        r"\b(hon'?ble|justice|mr\.?|mrs\.?|ms\.?)\b",
        "",
        name,
        flags=re.I
    )

    name = re.sub(
        r"\b(c\.?j\.?i?|j\.?j?\.?)\b",
        "",
        name,
        flags=re.I
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip(" ,.")

    return name.lower()


def extract_judges(text):

    hdr = text[:3000]

    seen = {}

    # ===================================================
    # Pattern 0
    # P.N. Bhagwati
    # M.H. Beg
    # B.K. Mukherjea
    # ===================================================

    for m in re.finditer(
        r"\b(?:[A-Z]\.){1,4}\s*[A-Z][a-z]{2,40}\b",
        hdr
    ):

        raw = m.group(0).strip()

        norm = _normalize_judge_name(raw)

        if len(norm) > 2:
            seen[norm] = raw

    # ===================================================
    # Pattern A
    # Hiralal J. Kania, C.J., Fazl Ali, J.
    # ===================================================

    for m in re.finditer(
        r"([A-Z][A-Za-z\.\s]{2,60}?)\s*,?\s*(?:C\.?J\.?I?|J\.?J?\.?)",
        hdr
    ):

        raw = m.group(1).strip()

        norm = _normalize_judge_name(raw)

        if len(norm) > 2:
            seen[norm] = raw.title()

    # ===================================================
    # Pattern B
    # CORAM block
    # ===================================================

    for cm in re.finditer(
        r"CORAM\s*:\s*(?:\d+\s+JUDGES?)?\s*\n([\s\S]{0,400})",
        hdr,
        re.I
    ):

        block = cm.group(1)

        for line in block.splitlines():

            line = line.strip()

            if re.search(r"\d", line):
                continue

            if len(line.split()) < 2:
                continue

            norm = _normalize_judge_name(line)

            if len(norm) > 2:
                seen[norm] = line.title()

    # ===================================================
    # Pattern C
    # Before: Hon'ble Justice X
    # ===================================================

    for m in re.finditer(
        r"(?:Before|BEFORE)\s*:\s*Hon['`]?(?:ble)?\s+(?:Mr\.?|Mrs\.?|Ms\.?)?\s*Justice\s+([A-Z][A-Za-z\s\.]{3,60})",
        hdr
    ):

        raw = m.group(1).strip()

        norm = _normalize_judge_name(raw)

        if len(norm) > 2:
            seen[norm] = raw.title()

    # ===================================================
    # Pattern D
    # Author:
    # ===================================================

    for m in re.finditer(
        r"Author\s*:\s*([A-Z][A-Za-z\s\.]{3,60})\s*,?\s*J\.?",
        hdr
    ):

        raw = m.group(1).strip()

        norm = _normalize_judge_name(raw)

        if len(norm) > 2:
            seen[norm] = raw.title()

    return list(seen.values())

# ============================================================
# METADATA
# ============================================================

def extract_metadata(text):

    hdr = text[:3000]

    date = None

    for pat, fmt in _DATE_PATTERNS:

        m = re.search(
            pat,
            hdr,
            re.I
        )

        if m:

            date = _to_iso(
                m.groups(),
                fmt
            )

            if date:
                break

    court = "Supreme Court of India"

    cm = re.search(
        r"(High Court of [A-Za-z\s]+)",
        hdr,
        re.I
    )

    if cm:

        court = (
            cm.group(1)
            .strip()
            .title()
        )

    judges = extract_judges(hdr)

    return {
        "date": date,
        "court": court,
        "judges": judges[:20],
    }


# ============================================================
# PARTY EXTRACTION
# ============================================================

def _extract_party_from_text(
    text,
    role
):

    m = re.search(
        rf"(?:{role}|appellant)[,\s]+([A-Z][A-Za-z\s\.]+?)(?:,|\s+(?:has|have|filed|submits|is|was)\b)",
        text[:500]
    )

    if m:
        return m.group(1).strip().title()

    return None


def extract_parties(
    filename,
    facts_text=""
):

    fn = re.sub(
        r"\.pdf$",
        "",
        filename,
        flags=re.I
    )

    # fix
    fn = re.sub(
        r"_{2,}",
        "_",
        fn
    )

    # fix
    fn = re.sub(
        r"\bM_[Ss]\b",
        "M/s",
        fn
    )

    # ===================================================
    # In Re
    # ===================================================

    if re.match(
        r"In_Re_",
        fn,
        re.I
    ):

        name = (
            fn[6:]
            .replace("_"," ")
            .strip()
            .title()
        )

        return (
            f"In Re {name}",
            None
        )

    # ===================================================
    # Suo Motu
    # ===================================================

    if re.match(
        r"Suo_Motu",
        fn,
        re.I
    ):

        return (
            "Court (Suo Motu)",
            None
        )

    # ===================================================
    # Standard vs
    # ===================================================

    m = re.search(
        r"(.+?)_vs?_(.+?)(?:__on_|_on_|\d{4}|$)",
        fn,
        re.I
    )

    if m:

        pet = (
            m.group(1)
            .replace("_"," ")
            .strip()
            .title()
        )

        res = (
            m.group(2)
            .replace("_"," ")
            .strip()
            .title()
        )

        pet = pet.replace(
            "M S ",
            "M/s "
        )

        return pet, res

    # ===================================================
    # Text fallback
    # ===================================================

    pet = _extract_party_from_text(
        facts_text,
        "petitioner"
    )

    res = _extract_party_from_text(
        facts_text,
        "respondent"
    )

    return pet, res


# ============================================================
# LEGAL CONCEPTS
# ============================================================

_CONCEPTS = {

    "natural_justice":
        ["natural justice","audi alteram partem"],

    "fundamental_rights":
        ["fundamental rights","article 14","article 19","article 21"],

    "due_process":
        ["due process","procedure established by law"],

    "mens_rea":
        ["mens rea","guilty mind"],

    "res_judicata":
        ["res judicata"],

    "habeas_corpus":
        ["habeas corpus"],

    "doctrine_of_severability":
        ["doctrine of severability","severable"],

    "ultra_vires":
        ["ultra vires"],

    "locus_standi":
        ["locus standi"],

    "writ_jurisdiction":
        ["certiorari","mandamus","writ jurisdiction"],

    "precedent":
        ["ratio decidendi","obiter dicta","stare decisis"],

    "fair_trial":
        ["fair trial","right to counsel"],

    "preventive_detention":
        ["preventive detention"],

    "promissory_estoppel":
        ["promissory estoppel","legitimate expectation"],

    "vicarious_liability":
        ["vicarious liability"],

    # NEW

    "beyond_reasonable_doubt":
        ["beyond reasonable doubt","benefit of doubt"],

    "circumstantial_evidence":
        ["circumstantial evidence","last seen theory"],

    "bail_jurisprudence":
        ["anticipatory bail","default bail"],

    "limitation":
        ["limitation period","cause of action"],

    "specific_performance":
        ["specific performance","breach of contract"],

    "tax_law":
        ["tax evasion","assessment order"],

    "service_law":
        ["service rules","departmental proceedings"],

    "property_law":
        ["adverse possession","title deed"],
        "double_jeopardy":
        ["double jeopardy"],

    "hostile_witness":
        ["hostile witness"],

    "dying_declaration":
        ["dying declaration"],

    "burden_of_proof":
        ["burden of proof"],

    "arbitration":
        ["arbitration", "arbitral award"],

    "consumer_protection":
        ["consumer protection", "consumer dispute"],
}


def extract_concepts(text):

    t = text.lower()

    return [
        k
        for k, kws in _CONCEPTS.items()
        if any(
            kw in t
            for kw in kws
        )
    ]


# ============================================================
# CASE ID
# ============================================================

def make_case_id(
    year,
    filename
):

    h = hashlib.md5(
        filename.encode()
    ).hexdigest()[:12]

    return f"{year}_{h}"


def title_from_filename(filename):

    fn = re.sub(
        r"\.pdf$",
        "",
        filename,
        flags=re.I
    )

    fn = re.sub(
        r"__on_.+$",
        "",
        fn
    )

    return (
        fn.replace("_"," ")
        .strip()
        .title()
    )


# ============================================================
# OUTCOME
# ============================================================

def regex_outcome(decision):

    t = decision.lower()

    # --------------------------------------------------
    # withdrawn
    # --------------------------------------------------

    if re.search(
        r"dismissed\s+as\s+withdrawn|withdrawn\s+by\s+(?:the\s+)?petitioner",
        t
    ):
        return "Unknown"

    # --------------------------------------------------
    # respondent objection dismissed
    # --------------------------------------------------

    if re.search(
        r"(?:respondent'?s?|objection|preliminary\s+objection).{0,40}?dismissed",
        t
    ):
        return "Accept"

    # --------------------------------------------------
    # strongest signals
    # --------------------------------------------------

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?(?:partly|partially)\s+allowed",
        t
    ):
        return "Partially Accept"

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?(?:allowed|accepted|granted)",
        t
    ):
        return "Accept"

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?dismissed",
        t
    ):
        return "Reject"

    # --------------------------------------------------
    # secondary signals
    # --------------------------------------------------

    if any(
        x in t
        for x in [
            "partly allowed",
            "partially allowed",
            "allowed in part",
            "modified"
        ]
    ):
        return "Partially Accept"

    if any(
        x in t
        for x in [
            "quashed",
            "set aside",
            "acquitted"
        ]
    ):
        return "Accept"

    if any(
        x in t
        for x in [
            "dismissed",
            "fails",
            "no merit"
        ]
    ):
        return "Reject"

    return "Unknown"

def safe_filename(case_id):

    return (
        re.sub(
            r"[^\w\-]",
            "_",
            case_id
        )[:120]
        + ".json"
    )


print("✅ Helpers ready")

✅ Helpers ready


CELL 7 — Load CSV

In [ ]:
# ── CELL 7: Load CSV ─────────────────────────────────────────

print("Loading CSV...")

df = pd.read_csv(
    FILE_PATH,
    dtype=str
)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# fix: confidence column detection
HAS_CONFIDENCE = (
    "confidence" in df.columns
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

# fix: lowercase filename for collision-proof case grouping
df["_casekey"] = (
    df["filename"]
    .fillna("unknown")
    .str.lower()
    + "__"
    + df["year"].astype(str)
)

all_keys = (
    df["_casekey"]
    .unique()
    .tolist()
)

print(
    f"✅ {len(df):,} rows | "
    f"{len(all_keys):,} unique cases"
)

if HAS_CONFIDENCE:
    print("✅ confidence column detected")
else:
    print(
        "⚠️ confidence column not found "
        "(confidence filtering disabled)"
    )

Loading CSV...
✅ 3,684,243 rows | 26,688 unique cases
✅ confidence column detected


CELL 8 — Checkpoint

In [ ]:
# ── CELL 8: Checkpoint ───────────────────────────────────────

CKPT_FILE = os.path.join(
    CHECKPOINT_DIR,
    "done_keys.json"
)

ERROR_FILE = os.path.join(
    CHECKPOINT_DIR,
    "errors.json"
)


def load_done() -> set:

    if (
        RESUME
        and os.path.exists(CKPT_FILE)
    ):

        try:

            with open(
                CKPT_FILE,
                "r",
                encoding="utf-8"
            ) as f:

                keys = set(
                    json.load(f)
                )

            print(
                f"↩ Resuming — "
                f"{len(keys):,} cases already completed"
            )

            return keys

        except Exception as e:

            print(
                "⚠️ Failed to load checkpoint:",
                e
            )

            return set()

    return set()


def save_done(keys: set):

    try:

        with open(
            CKPT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                list(keys),
                f,
                indent=2
            )

    except Exception as e:

        print(
            "⚠️ Failed to save checkpoint:",
            e
        )


def save_errors(errors):

    try:

        with open(
            ERROR_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                errors,
                f,
                indent=2,
                ensure_ascii=False
            )

    except Exception as e:

        print(
            "⚠️ Failed to save error log:",
            e
        )


def save_case(record: dict):

    """
    Save one JSON file per case.
    """

    fname = safe_filename(
        record["case_id"]
    )

    path = os.path.join(
        OUTPUT_DIR,
        fname
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            record,
            f,
            indent=2,
            ensure_ascii=False
        )


# fix: helper used when daily token limit reached
def save_checkpoint_and_stop(
    done_keys,
    errors=None
):

    print(
        "\n💾 Saving checkpoint..."
    )

    save_done(done_keys)

    if errors:
        save_errors(errors)

    print(
        "✅ Checkpoint saved."
    )


print("✅ Checkpoint ready")

✅ Checkpoint ready


CELL 9 — PART 1 (Helpers + Issue Hint + Beginning of process_case)

In [ ]:
# ── CELL 9: MAIN LOOP — PART 1 ───────────────────────────────

_ISSUE_PATTERNS = [

    r"[Tt]he (?:short |only |sole |main )?(?:question|point|issue|matter)\s+(?:that\s+)?(?:arises?|for(?:\s+our|this\s+[Cc]ourt(?:'s)?)?\s+(?:consideration|determination))\s*(?:is|was)\s*[:\-]?\s*([^.]{20,200}\.)",

    r"[Ww]hether\s+.{20,200}\?"
]


def extract_issue_hint(text):

    text = text[:5000]

    for pat in _ISSUE_PATTERNS:

        m = re.search(
            pat,
            text,
            re.I
        )

        if m:
            return m.group(0).strip()

    return None


# ============================================================
# SMART JOIN
# ============================================================

def smart_join(sentences):

    cleaned = [

        str(s)
        .strip()
        .rstrip(".")

        for s in sentences

        if str(s).strip()
    ]

    if not cleaned:
        return ""

    return ". ".join(cleaned) + "."


# ============================================================
# PROCESS CASE
# ============================================================

def process_case(group: pd.DataFrame):

    filename = str(
        group["filename"].iloc[0]
    )

    year = (
        int(group["year"].iloc[0])
        if pd.notna(
            group["year"].iloc[0]
        )
        else 0
    )

    # =======================================================
    # BUILD LABEL MAP
    # =======================================================

    lbl_map = {}

    for _, row in group.iterrows():

        # fix #8
        if HAS_CONFIDENCE:

            try:

                conf = float(
                    row.get(
                        "confidence",
                        1.0
                    )
                )

            except:
                conf = 1.0

            if conf < 0.65:
                continue

        label = str(
            row["predicted_label"]
        ).strip()

        txt = str(
            row["text"]
        ).strip()

        lbl_map.setdefault(
            label,
            []
        ).append(txt)

    # =======================================================
    # LABEL JOINER
    # =======================================================

    def join(*labels):

        vals = []

        for lbl in labels:

            vals.extend(
                lbl_map.get(
                    lbl,
                    []
                )
            )

        return smart_join(vals)

    # =======================================================
    # TEXT BLOCKS
    # =======================================================

    facts_text = join(
        "Facts"
    )

    petitioner_args_text = join(
        "Arguments of Petitioner"
    )

    respondent_args_text = join(
        "Arguments of Respondent"
    )

    reasoning_text = join(
        "Reasoning"
    )

    decision_text = join(
        "Decision"
    )

    full_text = smart_join([

        facts_text,

        petitioner_args_text,

        respondent_args_text,

        reasoning_text,

        decision_text
    ])

    # =======================================================
    # ISSUE HINT
    # =======================================================

    issue_hint = extract_issue_hint(
        full_text
    )

    # =======================================================
    # DETERMINISTIC EXTRACTION
    # =======================================================

    meta = extract_metadata(
        full_text
    )

    petitioner, respondent = (
        extract_parties(
            filename,
            facts_text
        )
    )

    statutes = extract_statutes(
        full_text
    )

    citations = extract_citations(
        full_text
    )

    concepts = extract_concepts(
        full_text
    )
    # =======================================================
    # BUILD PROMPT
    # =======================================================

    prompt = build_prompt(
        facts=facts_text,
        petitioner_args=petitioner_args_text,
        respondent_args=respondent_args_text,
        reasoning=reasoning_text,
        decision=decision_text,
        title_hint=title_from_filename(
            filename
        ),
        year=year,
        issue_hint=issue_hint
    )

    # =======================================================
    # GROQ EXTRACTION
    # =======================================================

    sem, extraction_method = call_groq(
        prompt
    )

    # =======================================================
    # REGEX FALLBACK
    # =======================================================

    if sem is None:

        extraction_method = "regex_fallback"

        sem = {

            "title":
                title_from_filename(
                    filename
                ),

            "court":
                meta["court"],

            "bench_strength":
                len(meta["judges"])
                if meta["judges"]
                else None,

            "case_type":
                None,

            "issue":
                None,

            "facts":
                [],

            "petitioner_arguments":
                [],

            "respondent_arguments":
                [],

            "reasoning_principles":
                [],

            "reasoning_summary":
                (
                    reasoning_text[:400]
                    if reasoning_text
                    else None
                ),

            "outcome":
                regex_outcome(
                    decision_text
                ),

            "decision_details":
                (
                    decision_text[:500]
                    if decision_text
                    else None
                ),

            "outcome_reasoning":
                None,

            "case_summary":
                None,
        }

    # =======================================================
    # OUTCOME NORMALIZATION
    # =======================================================

    groq_outcome = normalize_outcome(
        sem.get("outcome")
    )

    if groq_outcome:

        final_outcome = groq_outcome

    else:

        final_outcome = regex_outcome(
            decision_text
        )

    if not final_outcome:
        final_outcome = "Unknown"

    sem["outcome"] = final_outcome

    # =======================================================
    # BENCH STRENGTH CLEANUP
    # =======================================================

    bench_strength = sem.get(
        "bench_strength"
    )

    try:

        if bench_strength is not None:
            bench_strength = int(
                bench_strength
            )

    except:

        bench_strength = (
            len(meta["judges"])
            if meta["judges"]
            else None
        )

    if (
        bench_strength is None
        and meta["judges"]
    ):
        bench_strength = len(
            meta["judges"]
        )

    # =======================================================
    # OUTPUT RECORD
    # =======================================================

    record = {

        # ===============================================
        # PIPELINE INFO
        # ===============================================

        "extraction_method":
            extraction_method,

        "case_id":
            make_case_id(
                year,
                filename
            ),

        "year":
            year,

        "title":
            sem.get("title")
            or
            title_from_filename(
                filename
            ),

        "filename":
            filename,

        # ===============================================
        # METADATA
        # ===============================================

        "metadata": {

            "date":
                meta["date"],

            "court":
                sem.get("court")
                or
                meta["court"],

            "judges":
                meta["judges"],

            "bench_strength":
                bench_strength,

            "case_type":
                sem.get(
                    "case_type"
                ),
        },

        # ===============================================
        # PARTIES
        # ===============================================

        "parties": {

            "petitioner":
                petitioner,

            "respondent":
                respondent,
        },

        # ===============================================
        # ISSUE
        # ===============================================

        "issue":
            sem.get(
                "issue"
            ),

        # ===============================================
        # FACTS
        # ===============================================

        "facts":
            sem.get(
                "facts"
            ) or [],

        # ===============================================
        # ARGUMENTS
        # ===============================================

        "petitioner_arguments":
            sem.get(
                "petitioner_arguments"
            ) or [],

        "respondent_arguments":
            sem.get(
                "respondent_arguments"
            ) or [],

        # ===============================================
        # REASONING
        # ===============================================

        "reasoning": {

            "statutes":
                statutes,

            "principles":
                sem.get(
                    "reasoning_principles"
                ) or [],

            "summary":
                sem.get(
                    "reasoning_summary"
                ),
        },

        # ===============================================
        # DECISION
        # ===============================================

        "decision": {

            "outcome":
                final_outcome,

            "details":
                sem.get(
                    "decision_details"
                ),

            "outcome_reasoning":
                sem.get(
                    "outcome_reasoning"
                ),
        },

        # ===============================================
        # KNOWLEDGE GRAPH
        # ===============================================

        "legal_concepts":
            concepts,

        "citations":
            citations,

        # ===============================================
        # SUMMARY
        # ===============================================

        "summary":
            sem.get(
                "case_summary"
            ),
    }

    return record

  # ============================================================
# RUN
# ============================================================

done_keys = load_done()

# ==========================================
# TEST MODE
# ==========================================

total = len(all_keys)
quarter = total // 4
midpoint = total // 2
print(f"Total cases: {total:,}")
print(f"Q2 slice: index {quarter:,} → {midpoint:,}")
print(f"Q2 size: {midpoint - quarter:,} cases")
print(f"Already done in Q2: {sum(1 for k in all_keys[quarter:midpoint] if k in done_keys):,}")



q2_keys = all_keys[quarter:midpoint]  # your fixed Q2 slice

# Now filter out already-done cases within Q2
todo = [k for k in q2_keys if k not in done_keys]

session_done = 0

errors = []

show_first_err = True

stop_processing = False

print(
    f"⏳ Remaining: "
    f"{len(todo):,} cases "
    f"(from selected second half)"
)

for batch_start in range(
    0,
    len(todo),
    BATCH_SIZE
):

    if stop_processing:
        break

    if session_done >= DAILY_LIMIT:

        print(
            "\n⏸ Daily case limit reached."
        )

        break

    batch_keys = (
        todo[
            batch_start:
            batch_start + BATCH_SIZE
        ]
    )

    batch_df = (

        df[
            df["_casekey"].isin(
                set(batch_keys)
            )
        ]

        .copy()
    )

    for key in tqdm(

        batch_keys,

        desc=(
            f"Batch "
            f"{batch_start // BATCH_SIZE + 1}"
        ),

        leave=False
    ):

        try:

            group = batch_df[
                batch_df["_casekey"]
                == key
            ]

            if group.empty:

                errors.append({

                    "key": key,

                    "error":
                        "empty group"
                })

                continue

            record = process_case(
                group
            )

            save_case(
                record
            )

            done_keys.add(
                key
            )

            session_done += 1

        # ===================================================
        # DAILY LIMIT
        # ===================================================

        except DailyLimitReached:

            print(
                "\n🛑 Daily token limit reached."
            )

            save_checkpoint_and_stop(
                done_keys,
                errors
            )

            stop_processing = True

            break

        # ===================================================
        # CTRL+C
        # ===================================================

        except KeyboardInterrupt:

            print(
                "\n⏸ Interrupted."
            )

            save_checkpoint_and_stop(
                done_keys,
                errors
            )

            raise

        # ===================================================
        # NORMAL ERRORS
        # ===================================================

        except Exception:

            tb = traceback.format_exc()

            errors.append({

                "key":
                    key,

                "error":
                    tb
            })

            if show_first_err:

                print(

                    f"\n❌ First error "
                    f"({key}):\n"

                    f"{tb}"
                )

                show_first_err = False

    # =======================================================
    # BATCH CHECKPOINT
    # =======================================================

    save_done(
        done_keys
    )

    save_errors(
        errors
    )

    del batch_df

    gc.collect()

    remaining = (
        len(todo)
        - session_done
    )

    print(

        f"\nBatch "
        f"{batch_start // BATCH_SIZE + 1}"
        f" ✅\n"

        f"Session Cases: "
        f"{session_done:,}\n"

        f"Global Cases: "
        f"{len(done_keys):,}\n"

        f"Remaining: "
        f"{remaining:,}\n"

        f"Errors: "
        f"{len(errors):,}\n"

        # fix #22
        f"Tokens Used: "
        f"{total_tokens_used:,}"
        f" / "
        f"{MAX_TOKEN_BUDGET:,}"
    )

# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    f"\n✅ Done — "
    f"{session_done:,} JSON files written "
    f"to {OUTPUT_DIR}"
)

if errors:

    err_path = os.path.join(
        CHECKPOINT_DIR,
        "errors.json"
    )

    print(
        f"{len(errors):,} errors logged → "
        f"{err_path}"
    )

↩ Resuming — 13,488 cases already completed
Total cases: 26,688
Q2 slice: index 6,672 → 13,344
Q2 size: 6,672 cases
Already done in Q2: 6,496
⏳ Remaining: 176 cases (from selected second half)


Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

Batch 1 ✅
Session Cases: 100
Global Cases: 13,588
Remaining: 76
Errors: 0
Tokens Used: 270,358 / 480,000


Batch 2:   0%|          | 0/76 [00:00<?, ?it/s]


⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

⏳ Minute limit reached.
Waiting 65 seconds...

Batch 2 ✅
Session Cases: 176
Global Cases: 13,664
Remaining: 0
Errors: 0
Tokens Used: 474,954 / 480,000

✅ Done — 176 JSON files written to /content/drive/MyDrive/fact_layer_cases_v2


In [ ]:
print(CHECKPOINT_DIR)
print(OUTPUT_DIR)

/content/drive/MyDrive/fact_layer_cases_v2/checkpoints
/content/drive/MyDrive/fact_layer_cases_v2


CELL 10 — Verify

In [ ]:
# ── CELL 10: Verify — print one sample file ──────────────────

import glob

files = glob.glob(
    os.path.join(
        OUTPUT_DIR,
        "*.json"
    )
)

print(
    f"Total JSON files in output folder: "
    f"{len(files):,}\n"
)

if not files:

    print(
        "⚠️ No JSON files found."
    )

else:

    sample_file = files[0]

    print(
        f"Sample file:\n{sample_file}\n"
    )

    with open(
        sample_file,
        encoding="utf-8"
    ) as f:

        sample = json.load(f)

    print(
        json.dumps(
            sample,
            indent=2,
            ensure_ascii=False
        )
    )

    print(
        "\n=================================="
    )

    print(
        "Extraction Method:",
        sample.get(
            "extraction_method"
        )
    )

    print(
        "Court:",
        sample.get(
            "metadata",
            {}
        ).get(
            "court"
        )
    )

    print(
        "Bench Strength:",
        sample.get(
            "metadata",
            {}
        ).get(
            "bench_strength"
        )
    )

    print(
        "Case Type:",
        sample.get(
            "metadata",
            {}
        ).get(
            "case_type"
        )
    )

    print(
        "Outcome:",
        sample.get(
            "decision",
            {}
        ).get(
            "outcome"
        )
    )

    print(
        "Outcome Reasoning:",
        sample.get(
            "decision",
            {}
        ).get(
            "outcome_reasoning"
        )
    )

    print(
        "\n=================================="
    )

    print(
        f"Total Tokens Used: "
        f"{total_tokens_used:,}"
    )

Total JSON files in output folder: 6,748

Sample file:
/content/drive/MyDrive/fact_layer_cases_v2/1985_111b134315d8.json

{
  "extraction_method": "groq",
  "case_id": "1985_111b134315d8",
  "year": 1985,
  "title": "Dr. Mrs. Sushma Sharma etc. Etc. Vs State Of Rajasthan & Ors",
  "filename": "Dr_Mrs_Sushma_Sharma_Etc_Etc_vs_State_Of_Rajasthan_Ors_on_12_March_1985_1.PDF",
  "metadata": {
    "date": "1978-04-18",
    "court": "Supreme Court of India",
    "judges": [
      "The Single",
      "Constitution. The Single",
      "The Division Bench Set Aside The Decision Of The Single"
    ],
    "bench_strength": 3,
    "case_type": "Constitutional"
  },
  "parties": {
    "petitioner": "Dr Mrs Sushma Sharma Etc Etc",
    "respondent": "State Of Rajasthan Ors"
  },
  "issue": "Whether the criterion of continuous employment for absorption of temporary lecturers is valid?",
  "facts": [
    "For a long time since the inception of the University, there had been no regular selections and app